In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['datos_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [3]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient, UpdateOne
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("Limpieza de procesos completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Lizette"
META_DATOS = 500  
datos_finales = []   
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)

try:
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    # Usamos la URL principal del buscador de convocatorias
    driver.get("https://www.empleospublicos.cl/pub/convocatorias/convocatorias.aspx")
    
    # --- PAUSA PARA BÚSQUEDA MANUAL O CARGA INICIAL ---
    # Dado que Empleos Públicos requiere ingresar el término en el buscador,
    # el script escribirá "informática" y presionará buscar automáticamente.
    try:
        input_busqueda = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "txtFiltroPalabra")) # ID típico del buscador
        )
        input_busqueda.send_keys("informatica")
        
        btn_buscar = driver.find_element(By.ID, "btnBuscar")
        driver.execute_script("arguments[0].click();", btn_buscar)
        time.sleep(5) # Espera a que carguen los resultados
    except Exception as e:
        print("No se pudo usar el buscador automático, extrayendo ofertas generales...")

    nivel_pagina = 1

    while len(datos_finales) < META_DATOS:
        print(f"--- Procesando Pagina {nivel_pagina} (Datos capturados: {len(datos_finales)}) ---")
        time.sleep(8)

        # En empleospublicos, los resultados suelen estar en una tabla. Buscamos las filas (tr).
        # Ajustar el selector de la tabla si es necesario ('table.table tbody tr')
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table tbody tr"))
        )

        filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

        for fila in filas:
            if len(datos_finales) >= META_DATOS:
                break
                
            try:
                # Extraemos las columnas (td) de la fila actual
                columnas = fila.find_elements(By.TAG_NAME, "td")
                
                # Descartamos filas vacías o de encabezado
                if len(columnas) < 3:
                    continue

                # En empleospublicos, el cargo y la institución suelen estar en texto dentro de los td
                # El orden de las columnas puede variar, extraemos todo el texto de la fila para validación
                texto_fila = fila.text.lower()
                
                # 1. Título del cargo (Generalmente es un enlace o el primer texto fuerte)
                try:
                    titulo = columnas[1].text.strip() # Ajustar índice según la tabla real
                    if not titulo:
                        titulo = fila.find_element(By.TAG_NAME, "a").text.strip()
                except:
                    continue # Si no hay título, la fila no es válida

                # 8. Empresa (En este caso, la Institución Pública)
                try:
                    empresa = columnas[0].text.strip() # Generalmente la primera columna
                except:
                    empresa = "Institucion Publica"

                # 2. Modalidad de trabajo
                if "teletrabajo" in texto_fila:
                    modalidad = "Teletrabajo"
                else:
                    modalidad = "Presencial (Estándar Público)"

                # 7. Tipo de horario
                if "honorarios" in texto_fila or "part" in texto_fila:
                    horario = "Honorarios / Parcial"
                else:
                    horario = "Contrata / Planta (Jornada Completa)"

                # 5. Tipo de moneda y Renta
                if "renta" in texto_fila or "$" in texto_fila:
                    moneda = "CLP (Renta Bruta Publicada)"
                else:
                    moneda = "CLP (Grado EUS / Oculto)"

                # Guardado del diccionario con las 8 etiquetas requeridas
                datos_finales.append({
                    "titulo_cargo": titulo,             
                    "categoria_empleo": CATEGORIA_BUSQUEDA, 
                    "modalidad_trabajo": modalidad,     
                    "pais": "Chile",                    
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"), 
                    "tipo_moneda": moneda,              
                    "tipo_horario": horario,            
                    "empresa": empresa                  
                })
            except Exception as e:
                continue

        if len(datos_finales) >= META_DATOS:
            break

        # Intenta avanzar a la siguiente página
        try:
            # Empleos Públicos usa paginación ASP.NET (1, 2, 3, 4, Siguiente...)
            # Buscamos un enlace que contenga texto "Siguiente" o ">>"
            try:
                btn_sig = driver.find_element(By.PARTIAL_LINK_TEXT, "Siguiente")
            except:
                btn_sig = driver.find_element(By.PARTIAL_LINK_TEXT, ">>")
                
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
            nivel_pagina += 1
        except:
            print("No hay mas paginas disponibles.")
            break

    print(f"Extracción terminada: {len(datos_finales)} cargos.")

except Exception as e:
    print(f"Error en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["EmpleosPublicos"]

    if datos_finales:
        operaciones = []
        for d in datos_finales:
            filtro = {"titulo_cargo": d["titulo_cargo"], "empresa": d["empresa"]}
            nuevos_valores = {"$set": d}
            operaciones.append(UpdateOne(filtro, nuevos_valores, upsert=True))

        resultado = coleccion.bulk_write(operaciones)
        print(f"Operacion en BD: {resultado.upserted_count} nuevos creados, {resultado.modified_count} actualizados.")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")

Limpieza de procesos completada.
Error en Selenium: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location

No hay datos para guardar.
